# Agentic Alpha Cognitive Alpha SOXL v1

Translate arXiv:2511.18850 CogAlpha ideas into the repo src factor framework, combine Alpha158 with Codex-compatible OHLCV seed factors, and evaluate the selected features as an ML throttle overlay for SOXL.

In [1]:

from __future__ import annotations

import json
import shutil
import sys
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

warnings.filterwarnings('ignore', message='X does not have valid feature names')

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.backtest import evaluate_ml_position_candidates, evaluate_weighted_position_candidates
from src.data.qlib_native import QlibProviderConfig, alpha158_legacy_feature_config, build_alpha158_legacy_dataset
from src.factor import (
    build_canary_sprint_frame,
    build_canary_trend_component,
    build_cognitive_alpha_seed_candidates,
    build_column_alpha_candidates,
    build_session_effect_frame,
    build_signed_overnight_component,
    build_tsmom_stop_component,
    cognitive_alpha_codex_prompt_seed,
    evaluate_cognitive_alpha_candidates,
    materialize_cognitive_alpha_features,
)
from soxl_backtest.reference import replay_soxlbacktest_reference

EXPERIMENT = 'agentic_alpha_cognitive_alpha_soxl_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
manifest = json.loads((PROVIDER_URI / 'qlib_store_manifest.json').read_text(encoding='utf-8'))
SOURCE_START = '2010-03-11'
DATA_START = '2019-01-02'
SOURCE_END = str(manifest.get('calendar_end') or manifest.get('as_of'))
DATA_END = SOURCE_END
TRAIN_WINDOW = ('2019-01-02', '2024-12-31')
VALIDATION_WINDOW = ('2025-01-02', '2025-12-31')
TEST_WINDOW = ('2026-01-02', DATA_END)
LABEL_COL = 'label_return_5d'
DAILY_RETURN_COL = 'forward_return'
MAX_DRAWDOWN_FLOOR = -25.0
MAX_POSITION = 1.0
MIN_VALIDATION_TRADE_COUNT = 4
MIN_VALIDATION_AVG_ABS_POSITION = 0.10
MAX_VALIDATION_REVERSAL_RATIO = 0.40
COST_PER_TURNOVER = 0.0003
MAX_SELECTED_ALPHA_FEATURES = 32
ALPHA_LABEL_EXPR = 'Ref($close, -5) / $close - 1'

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

workflow_yaml = "\n".join([
    f"experiment: {EXPERIMENT}",
    "card_type: cognitive_alpha_soxl_throttle_search",
    "paper_reference:",
    "  arxiv_id: '2511.18850'",
    "  method: cognitive_alpha_mining_code_based_evolution",
    "qlib:",
    f"  provider_uri: {PROVIDER_URI.as_posix()}",
    f"  recorder_uri: {RECORDER_URI.as_posix()}",
    "  market: all",
    "  region: us",
    "segments:",
    "  train:",
    f"    start: {TRAIN_WINDOW[0]}",
    f"    end: {TRAIN_WINDOW[1]}",
    "  validation:",
    f"    start: {VALIDATION_WINDOW[0]}",
    f"    end: {VALIDATION_WINDOW[1]}",
    "  test:",
    f"    start: {TEST_WINDOW[0]}",
    f"    end: {TEST_WINDOW[1]}",
    "cognitive_alpha:",
    "  codex_candidate_interface: src.factor.CognitiveAlphaCandidate",
    "  quality_checker: no_future_leakage_output_stability_missing_distinct",
    "  metrics:",
    "    - IC",
    "    - ICIR",
    "    - RankIC",
    "    - RankICIR",
    "    - MutualInformation",
    "  alpha158: true",
    f"  max_selected_alpha_features: {MAX_SELECTED_ALPHA_FEATURES}",
    "ml_throttle_model:",
    f"  model_target_col: {LABEL_COL}",
    f"  return_col: {DAILY_RETURN_COL}",
    "  selection_segment: validation",
    "  report_segment: test",
    f"  max_position: {MAX_POSITION}",
    f"  max_validation_reversal_ratio: {MAX_VALIDATION_REVERSAL_RATIO}",
    "artifact_policy:",
    "  policy: slim_card_recorder_first",
    "  machine_entrypoint: summary.json",
    "",
])
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')


def build_base_throttle_frame() -> tuple[pd.DataFrame, list[str], list[str]]:
    qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
    daily_qlib_frame = D.features(
        ['SOXL', 'SOXX'],
        ['$open', '$high', '$low', '$close', '$volume', '$vix'],
        start_time=SOURCE_START,
        end_time=SOURCE_END,
        freq='day',
    )
    session_features = build_session_effect_frame(daily_qlib_frame, start=DATA_START, end=DATA_END, min_history=20, include_incomplete_latest=True)
    canary_features = build_canary_sprint_frame(daily_qlib_frame, start=DATA_START, end=DATA_END, include_incomplete_latest=True)
    components = [
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret10_th0_lowhv_q70_s1.0_stop0.02', start=DATA_START, end=DATA_END, score_name='soxx_ret10', threshold=0.0, regime_filter='lowhv', scale=1.0, stop_loss=0.02, lowhv_quantile=0.70),
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03', start=DATA_START, end=DATA_END, score_name='soxx_ret252', threshold=0.15, regime_filter='vix30', scale=0.75, stop_loss=0.03, lowhv_quantile=0.70),
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02', start=DATA_START, end=DATA_END, score_name='soxx_ret10', threshold=0.0, regime_filter='lowhv', scale=0.75, stop_loss=0.02, lowhv_quantile=0.70),
        build_canary_trend_component(canary_features, component='canary_trend60_vix26_autocorr_h3_s1.0', start=DATA_START, end=DATA_END, trend_threshold=0.06, vix_max=26.0, hold_days=3, require_positive_autocorr=True, scale=1.0),
        build_signed_overnight_component(session_features, component='signed_overnight_mom20_n0.002_soxl60_vix45_s0.5', start=DATA_START, end=DATA_END, neutral_band=0.002, scale=0.5, momentum_window=20, trend_filter='soxl60', vix_max=45.0),
    ]
    component_frame = pd.concat(components, ignore_index=True)
    component_exposure = component_frame.pivot_table(index='datetime', columns='component', values='exposure', aggfunc='sum').sort_index().fillna(0.0)

    normalized = daily_qlib_frame.reset_index()
    normalized['instrument'] = normalized['instrument'].astype(str).str.upper()
    normalized['datetime'] = pd.to_datetime(normalized['datetime']).dt.normalize()
    soxl = normalized.loc[normalized['instrument'] == 'SOXL', ['datetime', '$open', '$high', '$low', '$close', '$volume', '$vix']].sort_values('datetime')
    soxx = normalized.loc[normalized['instrument'] == 'SOXX', ['datetime', '$close', '$volume']].sort_values('datetime').rename(columns={'$close': 'soxx_close', '$volume': 'soxx_volume'})
    panel = soxl.merge(soxx, on='datetime', how='inner').rename(columns={'$open': 'soxl_open', '$high': 'soxl_high', '$low': 'soxl_low', '$close': 'soxl_close', '$volume': 'soxl_volume', '$vix': 'vix'}).sort_values('datetime').reset_index(drop=True)
    panel['forward_return'] = panel['soxl_close'].pct_change().shift(-1)
    panel['label_return_5d'] = panel['soxl_close'].shift(-5) / panel['soxl_close'] - 1.0
    for window in [1, 2, 3, 5, 10, 20, 40, 60, 120, 126, 200, 252]:
        panel[f'soxl_ret{window}'] = panel['soxl_close'].pct_change(window)
        panel[f'soxx_ret{window}'] = panel['soxx_close'].pct_change(window)
    for window in [10, 20, 40, 60, 120, 252]:
        panel[f'soxl_hv{window}'] = panel['soxl_close'].pct_change().rolling(window, min_periods=min(window, 20)).std() * np.sqrt(252)
    for window in [20, 60, 120, 252]:
        panel[f'soxl_dd{window}'] = panel['soxl_close'] / panel['soxl_close'].rolling(window, min_periods=min(window, 20)).max() - 1.0
    panel['vix_chg1'] = panel['vix'].pct_change()
    panel['vix_chg5'] = panel['vix'].pct_change(5)
    panel['volume_ratio20'] = panel['soxl_volume'] / panel['soxl_volume'].rolling(20, min_periods=10).mean() - 1.0

    frame = component_exposure.reset_index().merge(panel, on='datetime', how='inner')
    component_cols = list(component_exposure.columns)
    old_selected_model = json.loads((ROOT / 'research' / 'agentic_alpha_ensemble_search_v1' / 'selected_model.json').read_text(encoding='utf-8'))
    old_weights = {str(key): float(value) for key, value in old_selected_model['weights'].items()}
    frame['old_raw_target'] = frame[component_cols].mul(pd.Series({column: old_weights.get(column, 0.0) for column in component_cols}), axis=1).sum(axis=1)
    frame['old_raw_cap1'] = frame['old_raw_target'].clip(0.0, 1.0)
    frame['canary_base'] = frame['canary_trend60_vix26_autocorr_h3_s1.0'].clip(0.0, 1.0)

    constrained_selected = json.loads((ROOT / 'research' / 'agentic_alpha_constrained_weight_search_v1' / 'constrained_weight_selected_model.json').read_text(encoding='utf-8'))
    _, base_positions = evaluate_weighted_position_candidates(
        frame.loc[(frame['datetime'] >= pd.Timestamp(DATA_START)) & (frame['datetime'] <= pd.Timestamp(DATA_END))],
        candidates=[{'candidate': 'constrained_base', 'weights': constrained_selected['weights'], 'policy': constrained_selected['policy']}],
        component_cols=component_cols,
        return_col='forward_return',
        cost_per_turnover=COST_PER_TURNOVER,
    )
    frame = frame.merge(base_positions[['datetime', 'target_position']].rename(columns={'target_position': 'constrained_base'}), on='datetime', how='left')
    frame['constrained_base'] = frame['constrained_base'].fillna(0.0).clip(0.0, 1.0)
    frame['base_trend_gap'] = frame['constrained_base'] - frame['old_raw_cap1']
    frame['open'] = frame['soxl_open']
    frame['high'] = frame['soxl_high']
    frame['low'] = frame['soxl_low']
    frame['close'] = frame['soxl_close']
    frame['volume'] = frame['soxl_volume']

    base_feature_cols = component_cols + [
        'old_raw_target', 'old_raw_cap1', 'canary_base', 'constrained_base', 'base_trend_gap',
        'soxl_ret1', 'soxl_ret2', 'soxl_ret5', 'soxl_ret10', 'soxl_ret20', 'soxl_ret60',
        'soxx_ret10', 'soxx_ret20', 'soxx_ret60', 'soxx_ret120', 'soxx_ret252',
        'soxl_hv20', 'soxl_hv60', 'soxl_dd20', 'soxl_dd60', 'soxl_dd120',
        'vix', 'vix_chg1', 'vix_chg5', 'volume_ratio20',
    ]
    frame = frame.loc[(frame['datetime'] >= pd.Timestamp(DATA_START)) & (frame['datetime'] <= pd.Timestamp(DATA_END))]
    frame = frame.replace([np.inf, -np.inf], np.nan)
    frame['forward_return'] = pd.to_numeric(frame['forward_return'], errors='coerce').fillna(0.0)
    frame = frame.reset_index(drop=True)
    return frame, base_feature_cols, component_cols


def load_alpha158_features() -> tuple[pd.DataFrame, list[str], dict[str, Any]]:
    config = QlibProviderConfig(provider_uri=PROVIDER_URI, market='all', region='us')
    _ = build_alpha158_legacy_dataset(
        config,
        instruments=['SOXL'],
        label=ALPHA_LABEL_EXPR,
        train=TRAIN_WINDOW,
        valid=VALIDATION_WINDOW,
        test=TEST_WINDOW,
    )
    feature_fields, feature_names, feature_set = alpha158_legacy_feature_config()
    raw = D.features(['SOXL'], feature_fields, start_time=DATA_START, end_time=DATA_END, freq='day')
    alpha_frame = raw.reset_index()
    alpha_frame['datetime'] = pd.to_datetime(alpha_frame['datetime']).dt.normalize()
    alpha_frame['instrument'] = alpha_frame['instrument'].astype(str).str.upper()
    alpha_frame = alpha_frame.loc[alpha_frame['instrument'] == 'SOXL', ['datetime', *feature_fields]].copy()
    alpha_frame = alpha_frame.rename(columns={field: f'alpha158_{name}' for field, name in zip(feature_fields, feature_names)})
    alpha_frame = alpha_frame.drop_duplicates(subset=['datetime']).sort_values('datetime')
    renamed_feature_names = [f'alpha158_{column}' for column in feature_names]
    return alpha_frame, renamed_feature_names, feature_set


base_frame, base_feature_cols, component_cols = build_base_throttle_frame()
alpha_frame, alpha_feature_cols, alpha_feature_set = load_alpha158_features()
ml_frame = base_frame.merge(alpha_frame, on='datetime', how='left')

seed_candidates = build_cognitive_alpha_seed_candidates()
alpha_candidates = build_column_alpha_candidates(alpha_feature_cols, level='alpha158_baseline', agent='Alpha158')
all_alpha_candidates = [*seed_candidates, *alpha_candidates]
mining_frame = ml_frame.loc[ml_frame['datetime'] <= pd.Timestamp(VALIDATION_WINDOW[1])].copy()
alpha_report = evaluate_cognitive_alpha_candidates(
    mining_frame,
    candidates=all_alpha_candidates,
    label_col=LABEL_COL,
    generation=0,
    min_observations=180,
    max_missing_ratio=0.55,
    min_distinct_values=8,
    qualified_percentile=0.55,
    elite_percentile=0.75,
    max_abs_correlation=0.97,
    fallback_top_n=MAX_SELECTED_ALPHA_FEATURES,
)
selected_ranked = (
    alpha_report.metrics.loc[alpha_report.metrics['redundancy_status'].isin(['selected', 'selected_fallback'])]
    .sort_values(['selection_tier', 'selection_score', 'min_metric_percentile'], ascending=[True, False, False])
)
selected_feature_names = selected_ranked['factor'].astype(str).head(MAX_SELECTED_ALPHA_FEATURES).tolist()
if not selected_feature_names:
    selected_feature_names = (
        alpha_report.metrics.loc[alpha_report.metrics['quality_status'] == 'pass']
        .sort_values(['selection_score', 'min_metric_percentile'], ascending=[False, False])
        .head(min(MAX_SELECTED_ALPHA_FEATURES, 12))['factor']
        .astype(str)
        .tolist()
    )
selected_feature_frame = materialize_cognitive_alpha_features(ml_frame, all_alpha_candidates, selected_names=selected_feature_names)
for column in selected_feature_names:
    ml_frame[column] = selected_feature_frame[column].to_numpy()
feature_cols = base_feature_cols + selected_feature_names
ml_frame = ml_frame.replace([np.inf, -np.inf], np.nan)
ml_frame[DAILY_RETURN_COL] = pd.to_numeric(ml_frame[DAILY_RETURN_COL], errors='coerce').fillna(0.0)
ml_frame = ml_frame.reset_index(drop=True)

alpha_metrics_path = CARD / 'cognitive_alpha_metrics.csv'
alpha_corr_path = CARD / 'cognitive_alpha_correlation.csv'
alpha_summary_path = CARD / 'cognitive_alpha_summary.json'
alpha_metrics = alpha_report.metrics.copy()
alpha_metrics['used_in_throttle'] = alpha_metrics['factor'].isin(selected_feature_names)
alpha_metrics.to_csv(alpha_metrics_path, index=False)
alpha_report.correlation.to_csv(alpha_corr_path)
alpha_selected_alpha158_count = int(sum(name in alpha_feature_cols for name in selected_feature_names))
alpha_selected_seed_count = int(sum(name not in alpha_feature_cols for name in selected_feature_names))
alpha_summary = {
    **alpha_report.summary,
    'raw_selected_count': alpha_report.summary['selected_count'],
    'selected_count': int(len(selected_feature_names)),
    'selected_feature_names': selected_feature_names,
    'alpha158_candidate_count': int(len(alpha_candidates)),
    'seed_candidate_count': int(len(seed_candidates)),
    'alpha158_selected_count': alpha_selected_alpha158_count,
    'seed_selected_count': alpha_selected_seed_count,
    'codex_prompt_seed': cognitive_alpha_codex_prompt_seed(),
    'selection_scope': 'train_plus_validation_only',
    'alpha158_feature_set': alpha_feature_set,
}
alpha_summary_path.write_text(json.dumps(alpha_summary, indent=2, sort_keys=True), encoding='utf-8')
print(json.dumps({'rows': len(ml_frame), 'base_features': len(base_feature_cols), 'selected_alpha_features': selected_feature_names, 'alpha_summary': {k: alpha_summary[k] for k in ['candidate_count', 'selected_count', 'alpha158_selected_count', 'seed_selected_count']}}, indent=2, sort_keys=True))

position_specs: list[dict[str, Any]] = []
for base_position_col in ['constrained_base', 'old_raw_cap1', 'canary_base']:
    for low_probability, high_probability in [(0.35, 0.60), (0.40, 0.65), (0.45, 0.70), (0.50, 0.75)]:
        for min_throttle in [0.35, 0.55, 0.75, 0.90]:
            for rebalance in [0.10, 0.20, 0.25]:
                for min_hold_days in [2, 3]:
                    position_specs.append({
                        'position_mode': 'base_throttle',
                        'base_position_col': base_position_col,
                        'low_probability': low_probability,
                        'high_probability': high_probability,
                        'min_throttle': min_throttle,
                        'max_throttle': 1.0,
                        'trend_floor_col': 'soxl_ret60',
                        'trend_floor_threshold': 0.0,
                        'trend_floor_fraction': max(min_throttle, 0.55),
                        'throttle_curve': 'smoothstep',
                        'scale': 1.0,
                        'rebalance_threshold': rebalance,
                        'max_daily_change': 0.50,
                        'min_hold_days': min_hold_days,
                        'min_position': 0.0,
                        'max_position': MAX_POSITION,
                        'objective_min_avg_abs_position': MIN_VALIDATION_AVG_ABS_POSITION,
                        'objective_return_weight': 0.25,
                        'objective_return_to_drawdown_weight': 1.0,
                        'objective_exposure_weight': 4.0,
                        'objective_max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
                        'turnover_penalty': 0.50,
                        'reversal_penalty': 4.0,
                        'trade_penalty': 0.05,
                    })

model_specs = [
    {'model': 'logistic_regression'},
    {'model': 'hist_gradient_boosting_classifier'},
    {'model': 'extra_trees_classifier'},
    {'model': 'random_forest_classifier'},
    {'model': 'lgbm_classifier'},
    {'model': 'ridge'},
    {'model': 'hist_gradient_boosting_regressor'},
    {'model': 'extra_trees_regressor'},
    {'model': 'random_forest_regressor'},
    {'model': 'lgbm_regressor'},
]

candidate_metrics, daily_positions = evaluate_ml_position_candidates(
    ml_frame,
    feature_cols=feature_cols,
    model_specs=model_specs,
    train_window=TRAIN_WINDOW,
    validation_window=VALIDATION_WINDOW,
    test_window=TEST_WINDOW,
    position_specs=position_specs,
    return_col=DAILY_RETURN_COL,
    label_col=LABEL_COL,
    cost_per_turnover=COST_PER_TURNOVER,
    min_validation_trade_count=MIN_VALIDATION_TRADE_COUNT,
    min_validation_avg_abs_position=MIN_VALIDATION_AVG_ABS_POSITION,
    validation_max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    max_validation_reversal_ratio=MAX_VALIDATION_REVERSAL_RATIO,
)
selected_validation_row = candidate_metrics.loc[(candidate_metrics['selected']) & (candidate_metrics['segment'] == 'validation')].iloc[0].to_dict()
selected_test_row = candidate_metrics.loc[(candidate_metrics['selected']) & (candidate_metrics['segment'] == 'test')].iloc[0].to_dict()
oracle_score_test_row = candidate_metrics.loc[candidate_metrics['segment'] == 'test'].sort_values('selection_objective_score', ascending=False).iloc[0].to_dict()
oracle_return_test_row = candidate_metrics.loc[candidate_metrics['segment'] == 'test'].sort_values('return_pct', ascending=False).iloc[0].to_dict()

legacy_replay = replay_soxlbacktest_reference(start=TEST_WINDOW[0], end=TEST_WINDOW[1])
legacy_metrics = legacy_replay.metrics
constrained_summary = json.loads((ROOT / 'research' / 'agentic_alpha_constrained_weight_search_v1' / 'summary.json').read_text(encoding='utf-8'))
constrained_reference = constrained_summary['selected_candidate']
previous_factor_summary = json.loads((ROOT / 'research' / 'agentic_alpha_ml_factor_enhanced_throttle_v1' / 'summary.json').read_text(encoding='utf-8'))
previous_factor_test = previous_factor_summary['selected_candidate']['test']
previous_ml_summary = json.loads((ROOT / 'research' / 'agentic_alpha_ml_position_model_v1' / 'summary.json').read_text(encoding='utf-8'))
previous_ml_test = previous_ml_summary['selected_candidate']['test']


def jsonable(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [jsonable(item) for item in value]
    if isinstance(value, tuple):
        return [jsonable(item) for item in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        numeric = float(value)
        return None if not np.isfinite(numeric) else numeric
    if pd.isna(value):
        return None
    return value


def compact_metric_row(row: dict[str, Any]) -> dict[str, Any]:
    keys = [
        'candidate', 'model', 'model_task', 'model_target_col', 'return_col', 'position_mode', 'threshold', 'scale',
        'return_pct', 'max_drawdown_pct', 'return_to_drawdown', 'selection_objective_score', 'avg_abs_position',
        'max_position', 'min_position', 'days_above_100', 'days_below_0', 'total_turnover', 'trade_count',
        'next_day_reversal_count', 'next_day_reversal_ratio', 'avg_holding_days', 'selection_eligible',
        'selection_gate', 'params',
    ]
    return {key: jsonable(row.get(key)) for key in keys if key in row}

candidate_metrics_path = CARD / 'cognitive_alpha_candidate_metrics.csv'
selected_model_path = CARD / 'cognitive_alpha_selected_model.json'
benchmark_path = CARD / 'backtest_benchmark_comparison.csv'
failure_modes_path = CARD / 'failure_modes.md'
daily_positions_tmp = CARD / '.cognitive_alpha_daily_positions.csv.gz'
candidate_metrics_full_tmp = CARD / '.cognitive_alpha_candidate_metrics_full.csv.gz'

slim_candidate_metrics = pd.concat([
    candidate_metrics.loc[candidate_metrics['selected']],
    candidate_metrics.loc[candidate_metrics['segment'] == 'validation'].sort_values('selection_objective_score', ascending=False).head(80),
    candidate_metrics.loc[candidate_metrics['segment'] == 'validation'].sort_values('return_pct', ascending=False).head(80),
    candidate_metrics.loc[candidate_metrics['segment'] == 'test'].sort_values('selection_objective_score', ascending=False).head(80),
    candidate_metrics.loc[candidate_metrics['segment'] == 'test'].sort_values('return_pct', ascending=False).head(80),
], ignore_index=True).drop_duplicates(subset=['candidate', 'segment']).reset_index(drop=True)
slim_candidate_metrics.to_csv(candidate_metrics_path, index=False)
candidate_metrics.to_csv(candidate_metrics_full_tmp, index=False, compression='gzip')
daily_positions.to_csv(daily_positions_tmp, index=False, compression='gzip')
selected_model_payload = {
    'candidate': selected_test_row['candidate'],
    'selected_by': 'validation_selection_objective_score',
    'validation': compact_metric_row(selected_validation_row),
    'test': compact_metric_row(selected_test_row),
}
selected_model_path.write_text(json.dumps(selected_model_payload, indent=2, sort_keys=True), encoding='utf-8')

benchmark = pd.DataFrame([
    {'benchmark': 'selected_cognitive_alpha_validation_model', 'return_pct': float(selected_test_row['return_pct']), 'max_drawdown_pct': float(selected_test_row['max_drawdown_pct']), 'max_position': float(selected_test_row['max_position']), 'avg_abs_position': float(selected_test_row['avg_abs_position']), 'trade_count': int(selected_test_row['trade_count']), 'next_day_reversal_count': int(selected_test_row['next_day_reversal_count'])},
    {'benchmark': 'previous_factor_enhanced_throttle', 'return_pct': float(previous_factor_test['return_pct']), 'max_drawdown_pct': float(previous_factor_test['max_drawdown_pct']), 'max_position': float(previous_factor_test['max_position']), 'avg_abs_position': float(previous_factor_test['avg_abs_position']), 'trade_count': int(previous_factor_test['trade_count']), 'next_day_reversal_count': int(previous_factor_test['next_day_reversal_count'])},
    {'benchmark': 'previous_ml_position_validation_model', 'return_pct': float(previous_ml_test['return_pct']), 'max_drawdown_pct': float(previous_ml_test['max_drawdown_pct']), 'max_position': float(previous_ml_test['max_position']), 'avg_abs_position': float(previous_ml_test['avg_abs_position']), 'trade_count': int(previous_ml_test['trade_count']), 'next_day_reversal_count': int(previous_ml_test['next_day_reversal_count'])},
    {'benchmark': 'oracle_score_test_best_cognitive_alpha_candidate', 'return_pct': float(oracle_score_test_row['return_pct']), 'max_drawdown_pct': float(oracle_score_test_row['max_drawdown_pct']), 'max_position': float(oracle_score_test_row['max_position']), 'avg_abs_position': float(oracle_score_test_row['avg_abs_position']), 'trade_count': int(oracle_score_test_row['trade_count']), 'next_day_reversal_count': int(oracle_score_test_row['next_day_reversal_count'])},
    {'benchmark': 'oracle_return_test_best_cognitive_alpha_candidate', 'return_pct': float(oracle_return_test_row['return_pct']), 'max_drawdown_pct': float(oracle_return_test_row['max_drawdown_pct']), 'max_position': float(oracle_return_test_row['max_position']), 'avg_abs_position': float(oracle_return_test_row['avg_abs_position']), 'trade_count': int(oracle_return_test_row['trade_count']), 'next_day_reversal_count': int(oracle_return_test_row['next_day_reversal_count'])},
    {'benchmark': 'constrained_weight_reference', 'return_pct': float(constrained_reference['return_pct']), 'max_drawdown_pct': float(constrained_reference['max_drawdown_pct']), 'max_position': float(constrained_reference['max_position']), 'avg_abs_position': None, 'trade_count': int(constrained_reference['trade_count']), 'next_day_reversal_count': int(constrained_reference['next_day_reversal_count'])},
    {'benchmark': 'soxl_backtest_reference', 'return_pct': float(legacy_metrics['rulebase_return_pct']), 'max_drawdown_pct': -float(legacy_metrics['rulebase_max_drawdown_pct']), 'max_position': float(legacy_replay.daily['position'].max()), 'avg_abs_position': float(legacy_replay.daily['position'].abs().mean()), 'trade_count': int(legacy_metrics['trade_count']), 'next_day_reversal_count': None},
    {'benchmark': 'soxl_buyhold', 'return_pct': float(legacy_metrics['soxl_buyhold_return_pct']), 'max_drawdown_pct': None, 'max_position': 1.0, 'avg_abs_position': 1.0, 'trade_count': 1, 'next_day_reversal_count': None},
])
benchmark.to_csv(benchmark_path, index=False)

beats_factor_enhanced = float(selected_test_row['return_pct']) > float(previous_factor_test['return_pct'])
beats_previous_ml = float(selected_test_row['return_pct']) > float(previous_ml_test['return_pct'])
beats_soxlbacktest = float(selected_test_row['return_pct']) > float(legacy_metrics['rulebase_return_pct'])
beats_constrained = float(selected_test_row['return_pct']) > float(constrained_reference['return_pct'])
model_families = ['ExtraTreesClassifier', 'ExtraTreesRegressor', 'HistGradientBoostingClassifier', 'HistGradientBoostingRegressor', 'LGBMClassifier', 'LGBMRegressor', 'LogisticRegression', 'RandomForestClassifier', 'RandomForestRegressor', 'Ridge']
feature_groups = {
    'base_throttle_features': int(len(base_feature_cols)),
    'alpha158_selected': int(alpha_selected_alpha158_count),
    'cognitive_seed_selected': int(alpha_selected_seed_count),
    'enhanced_total': int(len(feature_cols)),
}
report_summary = {
    'workflow': EXPERIMENT,
    'goal_achieved': bool(beats_factor_enhanced and beats_soxlbacktest and beats_constrained),
    'candidate_count': int(candidate_metrics['candidate'].nunique()),
    'metric_rows': int(len(candidate_metrics)),
    'feature_count': int(len(feature_cols)),
    'feature_groups': feature_groups,
    'feature_cols': feature_cols,
    'selected_candidate': selected_model_payload,
    'oracle_score_test_best_candidate': compact_metric_row(oracle_score_test_row),
    'oracle_return_test_best_candidate': compact_metric_row(oracle_return_test_row),
    'previous_factor_enhanced_throttle_reference': previous_factor_test,
    'previous_ml_position_reference': previous_ml_test,
    'constrained_weight_reference': constrained_reference,
    'legacy_soxlbacktest_benchmark': legacy_metrics,
    'comparison': {
        'selected_vs_factor_enhanced_return_gap_pct': float(selected_test_row['return_pct'] - previous_factor_test['return_pct']),
        'selected_vs_previous_ml_return_gap_pct': float(selected_test_row['return_pct'] - previous_ml_test['return_pct']),
        'selected_vs_soxlbacktest_return_gap_pct': float(selected_test_row['return_pct'] - legacy_metrics['rulebase_return_pct']),
        'selected_vs_constrained_return_gap_pct': float(selected_test_row['return_pct'] - constrained_reference['return_pct']),
        'beats_factor_enhanced_throttle': bool(beats_factor_enhanced),
        'beats_previous_ml': bool(beats_previous_ml),
        'beats_soxlbacktest': bool(beats_soxlbacktest),
        'beats_constrained_weight_reference': bool(beats_constrained),
    },
    'model_families': model_families,
}
failure_modes_path.write_text(
    '# Failure Modes\n\n'
    '- CogAlpha-style screening is implemented as deterministic Codex-compatible candidate evaluation, not a full autonomous multi-generation agent loop.\n'
    '- Alpha158 and generated factors are selected on train+validation only; 2026 remains a holdout report window.\n'
    '- Single-asset SOXL has much weaker cross-sectional evidence than the paper datasets, so IC/RankIC are time-series approximations.\n'
    '- The selected throttle can still trail constrained_weight_reference if extra factors reduce participation during strong trend regimes.\n',
    encoding='utf-8',
)

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder.log_params(provider_uri=PROVIDER_URI.as_posix(), feature_count=int(len(feature_cols)), candidate_count=int(candidate_metrics['candidate'].nunique()), alpha158_candidate_count=int(len(alpha_candidates)), seed_candidate_count=int(len(seed_candidates)), selected_alpha_feature_count=int(len(selected_feature_names)), train_start=TRAIN_WINDOW[0], train_end=TRAIN_WINDOW[1], validation_start=VALIDATION_WINDOW[0], validation_end=VALIDATION_WINDOW[1], test_start=TEST_WINDOW[0], test_end=TEST_WINDOW[1])
    recorder.log_metrics(selected_test_return_pct=float(selected_test_row['return_pct']), selected_test_max_drawdown_pct=float(selected_test_row['max_drawdown_pct']), selected_test_avg_abs_position=float(selected_test_row['avg_abs_position']), selected_test_reversal_ratio=float(selected_test_row['next_day_reversal_ratio']), factor_enhanced_return_pct=float(previous_factor_test['return_pct']), constrained_weight_return_pct=float(constrained_reference['return_pct']), soxlbacktest_return_pct=float(legacy_metrics['rulebase_return_pct']))
    for path in [candidate_metrics_path, selected_model_path, benchmark_path, failure_modes_path, alpha_metrics_path, alpha_corr_path, alpha_summary_path]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_cognitive_alpha_soxl')
    recorder.log_artifact(daily_positions_tmp.as_posix(), artifact_path='agentic_alpha_cognitive_alpha_soxl')
    recorder.log_artifact(candidate_metrics_full_tmp.as_posix(), artifact_path='agentic_alpha_cognitive_alpha_soxl')
    recorder_id = recorder.id
    recorder_local_dir = Path(recorder.uri).resolve()

daily_positions_tmp.unlink(missing_ok=True)
candidate_metrics_full_tmp.unlink(missing_ok=True)
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
artifact_policy = {
    'policy': 'slim_card_recorder_first',
    'workflow_config': 'workflow.yaml',
    'primary_artifact': f'{EXPERIMENT}.ipynb',
    'machine_entrypoint': 'summary.json',
    'kept_sidecars': ['backtest_benchmark_comparison.csv', 'cognitive_alpha_candidate_metrics.csv', 'cognitive_alpha_selected_model.json', 'cognitive_alpha_metrics.csv', 'cognitive_alpha_correlation.csv', 'cognitive_alpha_summary.json', 'failure_modes.md'],
    'recorder_only_artifacts': {
        'predictions': 'artifacts/pred.pkl',
        'labels': 'artifacts/label.pkl',
        'signals': 'artifacts/signals.pkl',
        'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
        'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
        'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
        'daily_positions': 'artifacts/agentic_alpha_cognitive_alpha_soxl/.cognitive_alpha_daily_positions.csv.gz',
        'candidate_metrics_full': 'artifacts/agentic_alpha_cognitive_alpha_soxl/.cognitive_alpha_candidate_metrics_full.csv.gz',
    },
}
promotion_decision = 'promote_to_production_review' if beats_factor_enhanced and beats_soxlbacktest and beats_constrained else 'not_promoted_cognitive_alpha_under_constrained_weight'
summary = {
    'experiment': EXPERIMENT,
    'card_type': 'cognitive_alpha_soxl_throttle_search',
    'promotion_decision': promotion_decision,
    'paper_reference': {'arxiv_id': '2511.18850', 'title': 'Cognitive Alpha Mining via LLM-Driven Code-Based Evolution', 'url': 'https://arxiv.org/pdf/2511.18850'},
    'as_of': DATA_END,
    'market': 'all',
    'region': 'us',
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'data_range': [DATA_START, DATA_END],
    'segments': {'train': list(TRAIN_WINDOW), 'validation': list(VALIDATION_WINDOW), 'test': list(TEST_WINDOW)},
    'goal_gate': {'selection_segment': 'validation', 'report_segment': 'test', 'model_target_col': LABEL_COL, 'return_col': DAILY_RETURN_COL, 'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'max_position': MAX_POSITION, 'min_validation_trade_count': MIN_VALIDATION_TRADE_COUNT, 'min_validation_avg_abs_position': MIN_VALIDATION_AVG_ABS_POSITION, 'max_validation_reversal_ratio': MAX_VALIDATION_REVERSAL_RATIO, 'require_return_above_factor_enhanced_throttle': True, 'require_return_above_soxlbacktest': True, 'require_return_above_constrained_weight_reference': True},
    'cognitive_alpha_mining': alpha_summary,
    'feature_groups': feature_groups,
    'selected_candidate': selected_model_payload,
    'oracle_score_test_best_candidate': compact_metric_row(oracle_score_test_row),
    'oracle_return_test_best_candidate': compact_metric_row(oracle_return_test_row),
    'previous_factor_enhanced_throttle_reference': previous_factor_test,
    'previous_ml_position_reference': previous_ml_test,
    'constrained_weight_reference': constrained_reference,
    'legacy_soxlbacktest_benchmark': legacy_metrics,
    'cognitive_alpha_throttle_report': report_summary,
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'DatasetH', 'DataHandlerLP', 'Alpha158.get_feature_config', 'qlib.workflow.R.start'],
    'qlib_docs': {'data': 'https://qlib.readthedocs.io/en/latest/component/data.html', 'model': 'https://qlib.readthedocs.io/en/latest/component/model.html', 'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html'},
    'workflow_config': 'workflow.yaml',
    'artifact_policy': artifact_policy,
    'sidecars': {'backtest_benchmark_comparison': 'backtest_benchmark_comparison.csv', 'candidate_metrics': 'cognitive_alpha_candidate_metrics.csv', 'cognitive_alpha_correlation': 'cognitive_alpha_correlation.csv', 'cognitive_alpha_metrics': 'cognitive_alpha_metrics.csv', 'cognitive_alpha_summary': 'cognitive_alpha_summary.json', 'failure_modes': 'failure_modes.md', 'selected_model': 'cognitive_alpha_selected_model.json'},
    'recorder_artifacts': recorder_artifacts,
}
(CARD / 'summary.json').write_text(json.dumps(jsonable(summary), indent=2, sort_keys=True), encoding='utf-8')
conclusion = (
    '# 结论\n\n'
    f'- what_ran: 执行 {EXPERIMENT}，把 arXiv:2511.18850 的 CogAlpha 思路落到 `src.factor.cognitive_alpha`：Codex 可生成候选、质量闸门、IC/ICIR/RankIC/RankICIR/MI 筛选；再结合 Alpha158={len(alpha_candidates)} 个候选和 seed={len(seed_candidates)} 个 OHLCV 候选，用 qlib SOXL 数据训练 ML throttle。\n'
    f'- key_metrics: validation 选中 {selected_model_payload["candidate"]}；2026 test return={selected_model_payload["test"]["return_pct"]:.4f}%，MaxDD={selected_model_payload["test"]["max_drawdown_pct"]:.4f}%，avg_abs_position={selected_model_payload["test"]["avg_abs_position"]:.4f}，max_position={selected_model_payload["test"]["max_position"]:.4f}，trade_count={selected_model_payload["test"]["trade_count"]}，next_day_reversal_ratio={selected_model_payload["test"]["next_day_reversal_ratio"]:.4f}。\n'
    f'- factor_mining: selected_alpha_features={len(selected_feature_names)}，alpha158_selected={alpha_selected_alpha158_count}，cognitive_seed_selected={alpha_selected_seed_count}，selection_scope=train_plus_validation_only。\n'
    f'- benchmark: factor_enhanced_throttle return={previous_factor_test["return_pct"]:.4f}%，soxl_backtest return={legacy_metrics["rulebase_return_pct"]:.4f}%，constrained_weight_reference return={constrained_reference["return_pct"]:.4f}%，oracle_return_test_top={oracle_return_test_row["return_pct"]:.4f}%。\n'
    f'- promotion_decision: {promotion_decision}。没有超过 constrained_weight_reference 时，只作为架构和研究验证卡，不替代当前主仓位模型。\n'
    '- risks: CogAlpha 这里是确定性 Codex-compatible 框架，不是完整多代自动进化；SOXL 单标的 IC 是时间序列近似，弱于论文的横截面股票池证据；本研究面向交易复盘和风险控制，不构成买卖建议。\n'
    f'- next_action: 若后续继续，应把 Codex 候选生成接成多代候选池，并用 walk-forward 重新选择阈值。qlib recorder id={recorder_id}。\n'
)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(conclusion)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[61033:MainThread](2026-06-25 14:12:41,372) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61033:MainThread](2026-06-25 14:12:41,374) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61033:MainThread](2026-06-25 14:12:41,375) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[61033:MainThread](2026-06-25 14:12:41,465) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[61033:MainThread](2026-06-25 14:13:10,218) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_cognitive_alpha_soxl_v1.


[61033:MainThread](2026-06-25 14:13:10,220) INFO - qlib.workflow - [exp.py:258] - Experiment 612978384397461840 starts running ...


[61033:MainThread](2026-06-25 14:13:10,297) INFO - qlib.workflow - [recorder.py:345] - Recorder 2413d524bbba49acb5f2d53549a4befe starts running under Experiment 612978384397461840 ...


[61033:MainThread](2026-06-25 14:13:10,476) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


# 结论

- what_ran: 执行 agentic_alpha_cognitive_alpha_soxl_v1，把 arXiv:2511.18850 的 CogAlpha 思路落到 `src.factor.cognitive_alpha`：Codex 可生成候选、质量闸门、IC/ICIR/RankIC/RankICIR/MI 筛选；再结合 Alpha158=166 个候选和 seed=8 个 OHLCV 候选，用 qlib SOXL 数据训练 ML throttle。
- key_metrics: validation 选中 extra_trees_classifier_base_throttle_constrained_base_p0.35-0.6_t0.9-1_s1_h0.25_hold3；2026 test return=373.1605%，MaxDD=-21.7470%，avg_abs_position=0.4454，max_position=0.9482，trade_count=18，next_day_reversal_ratio=0.0000。
- factor_mining: selected_alpha_features=21，alpha158_selected=21，cognitive_seed_selected=0，selection_scope=train_plus_validation_only。
- benchmark: factor_enhanced_throttle return=188.7222%，soxl_backtest return=137.9962%，constrained_weight_reference return=571.2201%，oracle_return_test_top=380.2427%。
- promotion_decision: not_promoted_cognitive_alpha_under_constrained_weight。没有超过 constrained_weight_reference 时，只作为架构和研究验证卡，不替代当前主仓位模型。
- risks: CogAlpha 这里是确定性 Codex-compatible 框架，不是完整多代自动进化；SOXL 单标的 IC 是时间序列近